In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:

def BFGS(f, x0, itermax, tol):

    # Garantizamos un vector unidimensional (1D)
    x0 = np.array(x0, dtype=float).flatten()

    # Evaluación inicial
    fx0 = f(x0)

    if np.linalg.norm(fx0) > tol:
        flag = 1
    else:
        flag = 0

    orbita = [x0.copy()]
    xk = x0.copy()
    iter_count = 1
    error = 0.0

    # Matriz identidad B = eye(length(x0))
    B = np.eye(len(x0))

    while flag == 1 and iter_count < itermax:
        fx0 = f(x0)

        # Cálculo del paso sk = -B \ f(x0)
        sk = -np.linalg.solve(B, fx0)

        # Actualización de la posición
        xk = x0 + sk
        error = np.linalg.norm(fx0) #(xk - x0)

        # Criterio de salida
        if error < ((np.linalg.norm(x0) + 1.0) * tol):
            flag = 0

        orbita.append(xk.copy())
        iter_count += 1

        # Diferencia de gradientes y actualización de B
        fxk = f(xk)
        yk = fxk - fx0

        # Equivalente a: Bk = B + yk*yk'/(yk'*sk) - B*sk*sk'*B/(sk'*B*sk)
        term1 = np.outer(yk, yk) / np.dot(yk, sk)

        B_sk = B @ sk
        term2 = np.outer(B_sk, B_sk) / np.dot(sk, B_sk)

        B = B + term1 - term2
        x0 = xk.copy()

    return xk, orbita, error

In [ ]:
def line_search_backtracking(f_obj, grad, x, d, c1=1e-4, rho=0.5):

    alpha = 1.0
    g = grad(x)

    # Condición de Armijo: f(x + alpha*d) <= f(x) + c1 * alpha * (g^T d)
    while f_obj(x + alpha * d) > f_obj(x) + c1 * alpha * np.dot(g, d):
        alpha *= rho
        if alpha < 1e-12:  # Prevenir bucle infinito por alta rigidez
            break

    return alpha

def Gradiente_Conjugado_NL(f_obj, grad, x0, itermax=1000, tol=1e-6, method='PR'):

    x0 = np.array(x0, dtype=float).flatten()
    xk = x0.copy()
    orbita = [xk.copy()]

    gk = grad(xk)
    dk = -gk.copy()

    iter_count = 0

    while np.linalg.norm(gk) > tol and iter_count < itermax:
        # 1. Encontrar la longitud de paso alpha mediante búsqueda de línea
        alpha = line_search_backtracking(f_obj, grad, xk, dk)

        # 2. Actualizar la solución
        xk = xk + alpha * dk
        orbita.append(xk.copy())

        # 3. Evaluar el nuevo gradiente
        gk_next = grad(xk)

        # 4. Calcular el parámetro beta de conjugación
        if method == 'PR':
            # Polak-Ribière con reinicio automático (PR+)
            num = np.dot(gk_next, gk_next - gk)
            den = np.dot(gk, gk)
            beta = max(0.0, num / den)
        elif method == 'FR':
            # Fletcher-Reeves
            beta = np.dot(gk_next, gk_next) / np.dot(gk, gk)
        else:
            beta = 0.0

        # 5. Actualizar la dirección de búsqueda
        dk = -gk_next + beta * dk

        # 6. Reinicio a la dirección del gradiente si se pierde la propiedad de descenso
        if np.dot(gk_next, dk) >= 0:
            dk = -gk_next

        gk = gk_next
        iter_count += 1

    return xk, orbita, np.linalg.norm(gk)

In [ ]:
def Gauss_Newton(r, x0, itermax, tol, J=None):

    # Garantizar vector unidimensional de flotantes
    x0 = np.array(x0, dtype=float).flatten()
    xk = x0.copy()
    orbita = [xk.copy()]

    # Función interna para aproximación numérica de la Jacobiana (diferencias finitas)
    def jacobiano_numerico(x, eps=1e-8):
        rx = r(x)
        m = len(rx)
        n = len(x)
        J_mat = np.zeros((m, n))
        for j in range(n):
            x_step = x.copy()
            x_step[j] += eps
            J_mat[:, j] = (r(x_step) - rx) / eps
        return J_mat

    iter_count = 0
    error = np.linalg.norm(r(xk))

    while error > tol and iter_count < itermax:
        rx = r(xk)

        # Evaluar la matriz Jacobiana analítica o numérica
        Jx = J(xk) if J is not None else jacobiano_numerico(xk)

        # Resolver el sistema J(x) * s = -r(x) en sentido de mínimos cuadrados
        # np.linalg.lstsq ofrece mayor estabilidad numérica que la inversión explícita
        sk, _, _, _ = np.linalg.lstsq(Jx, -rx, rcond=None)

        # Actualización de la solución
        xk = xk + sk
        orbita.append(xk.copy())

        # Criterio de parada basado en la magnitud del paso
        if np.linalg.norm(sk) < tol * (np.linalg.norm(xk) + 1.0):
            error = np.linalg.norm(r(xk))
            break

        error = np.linalg.norm(r(xk))
        iter_count += 1

    return xk, orbita, error

In [ ]:
def Levenberg_Marquardt(r, x0, itermax=1000, tol=1e-6, lam=1e-3, v=10.0, J=None):

    x0 = np.array(x0, dtype=float).flatten()
    xk = x0.copy()
    orbita = [xk.copy()]
    n = len(xk)

    # Función interna para aproximación numérica de la Jacobiana
    def jacobiano_numerico(x, eps=1e-8):
        rx = r(x)
        m = len(rx)
        J_mat = np.zeros((m, n))
        for j in range(n):
            x_step = x.copy()
            x_step[j] += eps
            J_mat[:, j] = (r(x_step) - rx) / eps
        return J_mat

    iter_count = 0
    rx = r(xk)
    cost_k = 0.5 * np.dot(rx, rx)

    while np.linalg.norm(rx) > tol and iter_count < itermax:
        # Evaluar la Jacobiana analítica o numérica
        Jx = J(xk) if J is not None else jacobiano_numerico(xk)

        JtJ = Jx.T @ Jx
        Jtr = Jx.T @ rx

        # Sistema modificado: (J^T J + lambda * I) s = -J^T r
        A = JtJ + lam * np.eye(n)

        try:
            sk = np.linalg.solve(A, -Jtr)
        except np.linalg.LinAlgError:
            sk, _, _, _ = np.linalg.lstsq(A, -Jtr, rcond=None)

        # Probar el nuevo punto candidato
        x_new = xk + sk
        r_new = r(x_new)
        cost_new = 0.5 * np.dot(r_new, r_new)

        # Evaluar si hubo mejora en la función de costo
        if cost_new < cost_k:
            # Aceptar paso y reducir parámetro de amortiguación lambda
            xk = x_new
            rx = r_new
            cost_k = cost_new
            lam = lam / v
            orbita.append(xk.copy())

            # Criterio de parada secundario basado en la magnitud del paso
            if np.linalg.norm(sk) < tol * (np.linalg.norm(xk) + 1.0):
                break
        else:
            # Rechazar paso e incrementar parámetro lambda para favorecer descenso de gradiente
            lam = lam * v

        iter_count += 1

    return xk, orbita, np.linalg.norm(rx)

In [ ]:
def vconvergencia(l, solk,sol):
  err=np.zeros(l)
  for i in range(l):
    err[i]=np.log(np.linalg.norm(solk[i]-sol))
  m1, b1= np.polyfit(err[0:l-1], err[1: l], 1)
  return m1


In [ ]:
def radconv(opt_solver, f_obj, grad, x_sol,
            r_min=1e-4, r_max=1.0, num_radios=50,
            num_angulos=8, tol_max=1e-6, tol_sol=1e-3, itermax=int(1e3)):

    radios = np.linspace(r_min, r_max, num_radios)
    angulos = np.linspace(0, 2 * np.pi, num_angulos, endpoint=False)

    tasas_exito = []

    for r in radios:
        exitos = 0
        for theta in angulos:
            x0 = x_sol + np.array([r * np.cos(theta), r * np.sin(theta)])

            try:

                if opt_solver.__name__ == 'Gradiente_Conjugado_NL':
                    xk, _, _ = opt_solver(f_obj, grad, x0, itermax=itermax, tol=tol_max, method='PR')
                else:
                    xk, _, _ = opt_solver(grad, x0, itermax, tol_max)
                if np.linalg.norm(xk - x_sol) < tol_sol:
                    exitos += 1
            except Exception:
                pass

        tasa = exitos / num_angulos
        tasas_exito.append(tasa)

    radio_max = 0.0
    for r, tasa in zip(radios, tasas_exito):
        if tasa == 1.0:
            radio_max = r
        else:
            break

    return radio_max

In [ ]:
#Rosenbrock
f3=lambda X: 100*(X[1]-X[0]**2)**2+(1-X[0])**2
gradf3=lambda X: np.array([2*(X[0]-1)-400*X[0]*(X[1]-(X[0])**2), 200*(X[1]-(X[0])**2)])

rBFGS=radconv(BFGS, f3, gradf3, np.ones(2))
x0B = np.array([1.0+rBFGS/np.sqrt(2), 1.0+rBFGS/np.sqrt(2)])
xkB, orbitaB, errorB = BFGS(gradf3, x0B, int(1e3), 1e-6)
print('Solución BFGS:', xkB)
print('Error BFGS:', errorB)
print('Iteraciones BFGS:', len(orbitaB) - 1)
print("radio de convergencia", rBFGS)
print('Velocidad de Convergencia BFGS',vconvergencia(len(orbitaB), orbitaB, np.ones(2)))
print('')

RGC=radconv(Gradiente_Conjugado_NL, f3, gradf3, np.ones(2), 1, 60)
x0GC=np.array([1.0+RGC/np.sqrt(2), 1.0+RGC/np.sqrt(2)])
xkGC, orbitaGC, errorGC = Gradiente_Conjugado_NL(f3, gradf3, x0GC, itermax=int(1e3), tol=1e-6, method='PR')
print('Solución Gradiente Conjugado:', xkGC)
print('Error Gradiente Conjugado:', errorGC)
print('Iteraciones Gradiente Conjugado:', len(orbitaGC) - 1)
print("radio de convergencia",RGC)
print('Velocidad de Convergencia Gradiente Conjugado',vconvergencia(len(orbitaGC), orbitaGC, np.ones(2)))
print('')

RN=radconv(Gauss_Newton, f3, gradf3, np.ones(2), 1, 30)
x0N=np.array([1.0+RN/np.sqrt(2), 1.0+RN/np.sqrt(2)])
xkN, orbitaN, errorN = Gauss_Newton(gradf3, x0N, int(1e3), 1e-6)
print('Solución Gauss Newton:', xkN)
print('Error Gauss Newton:', errorN)
print('Iteraciones Gauss Newton:', len(orbitaN) - 1)
print('Radio de convergencia de Gauss Newton', RN)
print('Velocidad de Gauss Newton',vconvergencia(len(orbitaN), orbitaN, np.ones(2)))


print('')
RLM=radconv(Levenberg_Marquardt, f3, gradf3, np.ones(2), 1, 10)
x0LM=np.array([1.0+RLM/np.sqrt(2), 1.0+RLM/np.sqrt(2)])
xkLM, orbitaLM, errorLM = Levenberg_Marquardt(gradf3, x0LM, itermax=int(1e3), tol=1e-6)
print('Solución Levenberg Marquardt:', xkLM)
print('Error Levenberg Marquardt:', errorLM)
print('Iteraciones Levenberg Marquardt:', len(orbitaLM) - 1)
print('Radio de convergencia de Levenberg-Marquartd', RLM)
print('Velocidad de convergencia de Levenberg-Marquartd', vconvergencia(len(orbitaLM), orbitaLM, np.ones(2)))



Solución BFGS: [1. 1.]
Error BFGS: 1.7121871333405808e-06
Iteraciones BFGS: 32
radio de convergencia 0.10213061224489796
Velocidad de Convergencia BFGS 1.2707197679495115

Solución Gradiente Conjugado: [1.00000006 1.00000012]
Error Gradiente Conjugado: 9.919961318132588e-07
Iteraciones Gradiente Conjugado: 987
radio de convergencia 14.244897959183673
Velocidad de Convergencia Gradiente Conjugado 0.997582617310927

Solución Gauss Newton: [1. 1.]
Error Gauss Newton: 1.5506947897350824e-07
Iteraciones Gauss Newton: 8
Radio de convergencia de Gauss Newton 25.26530612244898
Velocidad de Gauss Newton 1.5774350207364292

Solución Levenberg Marquardt: [1. 1.]
Error Levenberg Marquardt: 4.707720323536496e-10
Iteraciones Levenberg Marquardt: 83
Radio de convergencia de Levenberg-Marquartd 1.183673469387755
Velocidad de convergencia de Levenberg-Marquartd 1.3387656443344085
